# Quantum Canary

Check your qubits before a job, watch them during it, and compare after.

Install: `pip install "quantum-canary[ibm]"`

In [1]:
import warnings
warnings.filterwarnings("ignore")

import quantum_canary as canary
from qiskit import QuantumCircuit, transpile

print(canary.banner())


    (o>
 \\_//)     QUANTUM CANARY  v1.0.0
  \_/_)     qubit health checks before, during and after your job
   _|_


## Backend

This uses `fake_fez`, a local copy of IBM's 156-qubit Heron device with its real calibration and noise. It needs no account. Set `USE_IBM = True` to use real hardware instead (your saved IBM Quantum account).

In [2]:
USE_IBM = False

if USE_IBM:
    from qiskit_ibm_runtime import QiskitRuntimeService
    backend = QiskitRuntimeService().backend("ibm_fez")
else:
    from qiskit_ibm_runtime.fake_provider import FakeProviderForBackendV2
    backend = FakeProviderForBackendV2().backend("fake_fez")

print(backend.name, backend.num_qubits, "qubits")

fake_fez 156 qubits


## 1. Before the job

Check the qubits you plan to use, plus a spare. A RED qubit is swapped for the spare. The report also says whether to turn on dynamical decoupling.

In [3]:
before = canary.check(backend, qubits=[13, 14], spares=[12])
before.print()

━━ Pre-job check ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Backend     fake_fez (fake backend, local) · job 8b235c44-8569-4eb6-a667-81c497f73bb6
  Priors      live calibration

  Qubit  Role    Health           T1         T2        T2*   Δω/2π kHz       ε_sx  DD
  13     layout  GREEN      275.3 µs   207.6 µs     199 µs     -0.0532   1.60e-04  —
  14     layout  GREEN      229.2 µs   44.64 µs   51.33 µs      -0.146   1.64e-04  —
  12     spare   GREEN      77.25 µs   77.23 µs    72.6 µs     +0.0314   2.97e-04  —

  Use layout  [13, 14]
  Decoupling  not needed on these qubits
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


## 2. During the job

Your circuits, transpiled onto the checked layout. Here: 20 Bell-state circuits, with a Canary round after every 10.

In [4]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
bell.measure_all()

circuits = [transpile(bell, backend, initial_layout=before.layout) for _ in range(20)]

run = canary.run_monitored(backend, circuits, before, interval=10)
run.print()

━━ Monitored run ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Circuits    20 of yours, a Canary round after every 10 · job c44a9af3-90e5-459f-8579-89060818d221

  Round  Circuits    q13      q14      
  0      0-9         GREEN    GREEN    
  1      10-19       GREEN    GREEN    

  Drift       none since round 0
  Verdict     all rounds GREEN
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


Your results come back untouched and in order:

In [5]:
print(len(run.results), "results")
print(run.results[0].data.meas.get_counts())

20 results
{'11': 508, '00': 502, '01': 7, '10': 7}


## 3. After the job

Check the same qubits again and compare.

In [6]:
after = canary.check(backend, qubits=before.layout)
canary.compare(before, after).print()

━━ Post-job drift ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Qubit  Before   After         z T1     z T2     z Δω   z ε_sx  Drifted
  13     GREEN    GREEN        -0.55    -0.45    -0.82    +0.52  —
  14     GREEN    GREEN        +0.24    -0.38    +0.31    -1.23  —

  Verdict     stable: the qubits did not change during the job
              drift = Holm-corrected two-sample z-test at |z| > 2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
